# BERTScore Evaluation

This notebook evaluates the output quality of all three approaches from my project: the zero-shot inference model, the fine-tuned model, and the RAG system.

I use BERTScore as my evaluation metric. Unlike simple word-matching metrics like BLEU, BERTScore compares the semantic meaning of two texts using contextual embeddings from a pre-trained BERT model. This makes it much better suited for legal language, where the same concept can be expressed in many different ways and exact word matches are rare.

For each model, I load its results CSV, merge it with the ground truth answers on question ID, and compute Precision, Recall, and F1. The F1 score is the primary metric used to compare the three approaches.

### Setup and Evaluation Function

I define a single reusable function `run_bertscore_eval` that handles the full evaluation pipeline for any model's results file. This avoids repeating the same logic three times and ensures all three models are evaluated consistently using identical settings.

The function merges the model's answers with the ground truth on the `id` column rather than assuming row order is consistent across files. This is important because some rows may have been skipped or errored during inference, which would shift the row alignment and produce incorrect score pairings if I merged by position instead of by ID.

In [7]:
!pip install bert-score evaluate torch
import pandas as pd          # Used to load, merge, and manipulate the results and reference CSVs
from bert_score import score  # Core BERTScore function that computes Precision, Recall, and F1
from evaluate import load     # HuggingFace evaluate library used to load the bertscore metric
import os                     # Used to check whether the required CSV files exist before loading

# Load the BERTScore metric once so it is not reloaded for each of the three evaluations
bert_metric = load("bertscore")

# Path to the ground truth file containing the correct answers for all 644 questions
reference_file = 'dataset_clean_fixed.csv'

def run_bertscore_eval(results_file, output_name, task_label):
    """
    Loads a model's results CSV, merges it with the ground truth on question ID,
    computes BERTScore Precision/Recall/F1, and saves the scored output to a new CSV.
    Returns the scored DataFrame, or None if files are missing or no rows matched.
    """
    print(f"Starting Evaluation: {task_label}")

    # Guard: check both files exist before attempting to load them
    if not os.path.exists(results_file) or not os.path.exists(reference_file):
        print(f"Error: Missing {results_file} or {reference_file}")
        return None

    # Load the model's results CSV (semicolon separated)
    results_df = pd.read_csv(results_file, sep = ";")

    # Load the ground truth reference CSV (also semicolon separated)
    reference_df = pd.read_csv(reference_file, sep = ';')

    # Merge on 'id' so each model answer is paired with the correct ground truth answer.
    # This is safer than merging by row position, which breaks if any rows were skipped during inference.
    df = results_df.merge(reference_df, on = "id")

    # Drop any rows where either the model answer or the ground truth is missing
    df = df.dropna(subset = ["answer", "correct_answer"])

    # If no rows remain after merging and dropping, the ID columns likely do not match
    if df.empty:
        print(f"Warning: No matching data for {task_label}. Check CSV 'id' columns.")
        return None

    # Convert the answer and reference columns to plain Python lists as required by BERTScore
    predictions = df['answer'].fillna("").tolist()  # fillna ensures no NaN slips through
    references = df['correct_answer'].tolist()

    print(f"Evaluating {len(predictions)} samples...")

    # Compute BERTScore using the German BERT model.
    # lang='de' selects the German-optimized BERT checkpoint.
    # rescale_with_baseline=True shifts the raw scores so 0 means random and 1 means perfect,
    # making the numbers more intuitive to interpret and compare across models.
    P, R, F1 = score(
        predictions,
        references,
        lang = "de",
        rescale_with_baseline = True,
        verbose = False
    )

    # Compute the mean across all samples.
    # .item() converts a single-element PyTorch tensor to a plain Python float.
    mean_p = P.mean().item()
    mean_r = R.mean().item()
    mean_f1 = F1.mean().item()

    # Attach the per-sample scores back to the DataFrame for detailed analysis
    df['bert_p'] = P.tolist()
    df['bert_r'] = R.tolist()
    df['bert_f1'] = F1.tolist()

    # Attach the overall mean F1 to every row so it is easy to look up after loading the CSV
    df['avg_f1_score'] = mean_f1

    # Print the summary scores to the console
    print(f"Results for {task_label}:")
    print(f"Precision: {mean_p:.4f}")
    print(f"Recall: {mean_r:.4f}")
    print(f"F1 Score: {mean_f1:.4f}\n")

    # Save the full per-sample scored DataFrame to a new CSV for later review
    df.to_csv(output_name, index = False)

    return df

# Run evaluation for all three models

# 1. Zero-Shot Inference: GPT-4o-mini with no training or retrieval
inference_eval = run_bertscore_eval(
    'inference_results.csv',
    'inference_bertscore.csv',
    'Zero-Shot Inference Model'
)

# 2. Fine-Tuned Model: Llama 3.1 8B fine-tuned on Austrian tax law via LoRA
finetuned_eval = run_bertscore_eval(
    'fine_tuned_results.csv',
    'fine_tuned_bertscore.csv',
    'Fine-Tuned Model'
)

# 3. RAG System: Gemma 1.1 2B with retrieved legal context from the tax law PDF
rag_eval = run_bertscore_eval(
    'rag_results.csv',
    'rag_bertscore.csv',
    'RAG Model'
)


Starting Evaluation: Zero-Shot Inference Model
Evaluating 643 samples...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Results for Zero-Shot Inference Model:
Precision: 0.0537
Recall: 0.3087
F1 Score: 0.1718

Starting Evaluation: Fine-Tuned Model
Evaluating 643 samples...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Results for Fine-Tuned Model:
Precision: 0.3236
Recall: 0.2437
F1 Score: 0.2812

Starting Evaluation: RAG Model
Evaluating 643 samples...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Results for RAG Model:
Precision: 0.2706
Recall: 0.1764
F1 Score: 0.2211

